# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** Sagyntay Aiziya

**Student ID:** S23068400

**Group:** 9

**Repository:** https://github.com/aiziya-sagyntay/css4007-hw3

## AI tool disclosure

> I used ChatGPT to better understand the assignment requirements and clarify concepts and instructions.

Run this notebook **top to bottom** before you push. A cell with no saved output earns nothing.
The instructions for every section are in `README.md`.

---
# Part A — relevance by hand (1 pt)

| | Text |
|---|---|
| D1 | Large language models can generate text and answer questions. |
| D2 | Retrieval-Augmented Generation retrieves relevant documents before generating an answer. |
| D3 | Embeddings represent text as numerical vectors. |
| Query | How does RAG retrieve information? |

`k1 = 1.5`, `b = 0.75`, `idf(t) = ln((N − n(t) + 0.5) / (n(t) + 0.5))`. Tokenise as README §3 says.

## A1 (a) — BM25 as written

**Tokens of each document, and its length:**


| Doc | Tokens | \|d\| |
|---|---|---|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |




**avgdl =** (9 + 10 + 6) / 3 = 25 / 3 = 8.3333


**Query terms:** how, does, rag, retrieve, information


**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| does | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | 1.9459 |
| information | 0 | 0 | 0 | 0 | 1.9459 |

**IDF calculation:**

N = 3

idf(t) = ln((N - n(t) + 0.5) / (n(t) + 0.5))

For all query terms, n(t) = 0.

idf(t) = ln((3 - 0 + 0.5) / (0 + 0.5)) = ln(7) = 1.9459

**BM25 of each document:**

k1 = 1.5, b = 0.75, avgdl = 8.3333

BM25(D, Q) = sum of [idf(t) * f(t,D) * (k1 + 1) / (f(t,D) + k1 * (1 - b + b * |D| / avgdl))]

**D1:** All query-term frequencies are zero.

BM25(D1) = 0 + 0 + 0 + 0 + 0 = 0

**D2:** All query-term frequencies are zero.

BM25(D2) = 0 + 0 + 0 + 0 + 0 = 0

**D3:** All query-term frequencies are zero.

BM25(D3) = 0 + 0 + 0 + 0 + 0 = 0

| Document | BM25 Score | Rank |
|---|---:|---|
| D1 | 0.0000 | 1 (tie) |
| D2 | 0.0000 | 1 (tie) |
| D3 | 0.0000 | 1 (tie) |

**Conclusion:**

All three documents receive a BM25 score of zero because none of the query terms match the document tokens exactly. Therefore, all documents are tied, and BM25 cannot identify a unique first document.



## A1 (b) — BM25 with the stems from README §3

**Stemmed tokens of each document, and its length:**

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, gener, text, and, answer, questions | 9 |
| D2 | retriev, augmented, gener, retriev, relevant, documents, before, gener, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

**avgdl =** (9 + 10 + 6) / 3 = 25 / 3 = 8.3333

**Stemmed query terms:** how, doe, rag, retriev, inform

**TF, DF and IDF of each query term:**

| Query term | f(t, D1) | f(t, D2) | f(t, D3) | n(t) (DF) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | 2 | 0 | 1 | 0.5108 |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

**IDF calculation:**

N = 3

For "retriev", n(t) = 1.

idf(retriev) = ln((3 - 1 + 0.5) / (1 + 0.5))

idf(retriev) = ln(2.5 / 1.5) = 0.510826

For all other query terms, n(t) = 0.

idf(t) = ln(7) = 1.945910

**BM25 of each document, worked:**

k1 = 1.5, b = 0.75, avgdl = 8.3333

**D1:**

No stemmed query terms match D1.

BM25(D1) = 0 + 0 + 0 + 0 + 0 = 0

**D2:**

Only "retriev" matches, with f(t,D2) = 2.

BM25(D2) = 0.510826 * (2 * 2.5) / (2 + 1.5 * (0.25 + 0.75 * 10 / 8.3333))

BM25(D2) = 0.510826 * 5 / 3.725

BM25(D2) = 0.685672

**D3:**

No stemmed query terms match D3.

BM25(D3) = 0 + 0 + 0 + 0 + 0 = 0

**Final ranking:**

| Document | BM25 Score | Rank |
|---|---:|---|
| D1 | 0.000000 | 2 (tie) |
| D2 | 0.685672 | 1 |
| D3 | 0.000000 | 2 (tie) |

**Which document does BM25 retrieve first, in (a) and in (b)? Why?**

In A1(a), all documents have a BM25 score of zero because there are no exact word matches. Therefore, all three documents are tied.

In A1(b), D2 ranks first because stemming converts "retrieval" and "retrieves" into "retriev", which matches the query term. D2 receives a BM25 score of 0.685672, while D1 and D3 remain at zero.



## A2 — cosine similarity

**Given vectors:**

Query = [0.8, 0.2, 0.5]

D1 = [0.7, 0.1, 0.4]

D2 = [0.9, 0.3, 0.6]

D3 = [0.1, 0.8, 0.2]

**Cosine similarity formula:**

Cosine(Q, D) = (Q · D) / (norm(Q) * norm(D))

**Step 1. Dot products:**

Q · D1 = 0.8(0.7) + 0.2(0.1) + 0.5(0.4) = 0.78

Q · D2 = 0.8(0.9) + 0.2(0.3) + 0.5(0.6) = 1.08

Q · D3 = 0.8(0.1) + 0.2(0.8) + 0.5(0.2) = 0.34

**Step 2. Vector magnitudes:**

norm(Q) = sqrt(0.8² + 0.2² + 0.5²) = 0.964365

norm(D1) = sqrt(0.7² + 0.1² + 0.4²) = 0.812404

norm(D2) = sqrt(0.9² + 0.3² + 0.6²) = 1.122497

norm(D3) = sqrt(0.1² + 0.8² + 0.2²) = 0.830662

**Step 3. Cosine calculations:**

Cosine(Q, D1) = 0.78 / (0.964365 * 0.812404) = 0.995591

Cosine(Q, D2) = 1.08 / (0.964365 * 1.122497) = 0.997693

Cosine(Q, D3) = 0.34 / (0.964365 * 0.830662) = 0.424437

| Pair | Dot product (worked) | norm(Q) | norm(D) | Cosine |
|---|---|---:|---:|---:|
| Query · D1 | 0.56 + 0.02 + 0.20 = 0.78 | 0.964365 | 0.812404 | 0.995591 |
| Query · D2 | 0.72 + 0.06 + 0.30 = 1.08 | 0.964365 | 1.122497 | 0.997693 |
| Query · D3 | 0.08 + 0.16 + 0.10 = 0.34 | 0.964365 | 0.830662 | 0.424437 |

**Final ranking:**

| Document | Cosine Similarity | Rank |
|---|---:|---|
| D1 | 0.995591 | 2 |
| D2 | 0.997693 | 1 |
| D3 | 0.424437 | 3 |

**1. Which document is most similar to the query?**

D2 is the most similar document, with a cosine similarity of 0.997693.

**2. Is the ranking the same as BM25's? Compare it with both A1 rankings.**

No. In A1(a), all three documents are tied with a score of zero. In A1(b), D2 ranks first, while D1 and D3 are tied. Cosine similarity gives a complete ranking: D2 > D1 > D3. Therefore, D2 ranks first in both A1(b) and A2, but the overall rankings are different.

**3. Why can BM25 and embedding retrieval disagree?**

BM25 relies on word matches, so without stemming it cannot match "retrieve" with "retrieves" or "retrieval". Embedding retrieval uses numerical vectors to represent semantic similarity. D2 discusses retrieving documents, which is closely related to the query. D1 discusses language models answering questions, while D3 describes numerical representations of text. Therefore, embedding retrieval can rank documents differently even when exact query words are missing.


Optional: check your Part A arithmetic here. The working above must still be written out by hand.

In [2]:

import math
import re

docs = {
    "D1": "Large language models can generate text and answer questions.",
    "D2": "Retrieval-Augmented Generation retrieves relevant documents before generating an answer.",
    "D3": "Embeddings represent text as numerical vectors."
}

query = "How does RAG retrieve information?"

stems = {
    "retrieve": "retriev",
    "retrieves": "retriev",
    "retrieval": "retriev",
    "generate": "gener",
    "generating": "gener",
    "generation": "gener",
    "information": "inform",
    "does": "doe"
}

def tokenize(text, stemming=False):
    words = re.findall(r"[a-z]+", text.lower())
    return [stems.get(w, w) for w in words] if stemming else words

def check_bm25(stemming):
    documents = {d: tokenize(t, stemming) for d, t in docs.items()}
    terms = tokenize(query, stemming)
    avgdl = sum(len(w) for w in documents.values()) / 3
    scores = {}

    print("\nA1(b) STEMMED" if stemming else "\nA1(a) ORIGINAL")
    print("Document lengths:", {d: len(w) for d, w in documents.items()})
    print("avgdl:", round(avgdl, 4))
    print("Query terms:", terms)
    print("Term         D1  D2  D3  DF    IDF")

    for term in terms:
        counts = [documents[d].count(term) for d in docs]
        df = sum(f > 0 for f in counts)
        idf = math.log((3 - df + 0.5) / (df + 0.5))
        print(f"{term:<13}{counts[0]:<4}{counts[1]:<4}{counts[2]:<4}{df:<6}{idf:.4f}")

    for d, words in documents.items():
        score = 0
        for term in terms:
            f = words.count(term)
            df = sum(term in w for w in documents.values())
            idf = math.log((3 - df + 0.5) / (df + 0.5))
            score += idf * f * 2.5 / (
                f + 1.5 * (0.25 + 0.75 * len(words) / avgdl)
            )
        scores[d] = score

    print("BM25 scores:", {d: round(s, 6) for d, s in scores.items()})

check_bm25(False)
check_bm25(True)

print("\nA2 COSINE SIMILARITY")

Q = [0.8, 0.2, 0.5]
vectors = {
    "D1": [0.7, 0.1, 0.4],
    "D2": [0.9, 0.3, 0.6],
    "D3": [0.1, 0.8, 0.2]
}

norm_q = math.sqrt(sum(x*x for x in Q))
print("Document | Dot product | norm(Q) | norm(D) | Cosine")

for name, D in vectors.items():
    dot = sum(x*y for x, y in zip(Q, D))
    norm_d = math.sqrt(sum(x*x for x in D))
    cosine = dot / (norm_q * norm_d)
    print(f"{name} | {dot:.4f} | {norm_q:.6f} | {norm_d:.6f} | {cosine:.6f}")



A1(a) ORIGINAL
Document lengths: {'D1': 9, 'D2': 10, 'D3': 6}
avgdl: 8.3333
Query terms: ['how', 'does', 'rag', 'retrieve', 'information']
Term         D1  D2  D3  DF    IDF
how          0   0   0   0     1.9459
does         0   0   0   0     1.9459
rag          0   0   0   0     1.9459
retrieve     0   0   0   0     1.9459
information  0   0   0   0     1.9459
BM25 scores: {'D1': 0.0, 'D2': 0.0, 'D3': 0.0}

A1(b) STEMMED
Document lengths: {'D1': 9, 'D2': 10, 'D3': 6}
avgdl: 8.3333
Query terms: ['how', 'doe', 'rag', 'retriev', 'inform']
Term         D1  D2  D3  DF    IDF
how          0   0   0   0     1.9459
doe          0   0   0   0     1.9459
rag          0   0   0   0     1.9459
retriev      0   2   0   1     0.5108
inform       0   0   0   0     1.9459
BM25 scores: {'D1': 0.0, 'D2': 0.685672, 'D3': 0.0}

A2 COSINE SIMILARITY
Document | Dot product | norm(Q) | norm(D) | Cosine
D1 | 0.7800 | 0.964365 | 0.812404 | 0.995591
D2 | 1.0800 | 0.964365 | 1.122497 | 0.997693
D3 | 0.3400 | 0

---
# Part B — the same two retrievers, from libraries (1.5 pt)

Load `data/corpus.jsonl` and `data/queries.json`.

In [3]:

import json
from pathlib import Path
import subprocess

# Repository location
repo = Path("/content/css4007-hw3")

# Clone repository if necessary
if not repo.exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/aiziya-sagyntay/css4007-hw3.git",
        str(repo)
    ], check=True)

# Load corpus
with open(repo / "data/corpus.jsonl", encoding="utf-8") as f:
    corpus = [json.loads(line) for line in f if line.strip()]

# Load queries
with open(repo / "data/queries.json", encoding="utf-8") as f:
    query_data = json.load(f)

queries = query_data["queries"]

# Check loaded data
print("Total chunks:", len(corpus))
print("Total queries:", len(queries))
print("First chunk ID:", corpus[0]["id"])
print("First query:", queries[0]["text"])


Total chunks: 74
Total queries: 8
First chunk ID: KZ-01
First query: What is form GR-305 used for?


## B1 — BM25 with `rank_bm25`


**My tokenisation:**

I converted all text to lowercase and used the regular expression `\w+` to split text into tokens and remove punctuation. I did not apply stemming or remove stop words.


In [4]:
%pip install -q rank-bm25

In [5]:
import re
from rank_bm25 import BM25Okapi

# Tokenization
def tokenize(text):
    return re.findall(r"\w+", text.lower())

# Prepare documents
documents = [chunk["text"] for chunk in corpus]
tokenized_docs = [tokenize(text) for text in documents]

# Build BM25 index
bm25 = BM25Okapi(tokenized_docs, k1=1.5, b=0.75)

# Search function
def bm25_search(query, top_k=3):
    scores = bm25.get_scores(tokenize(query))
    top_indices = sorted(
        range(len(scores)),
        key=lambda i: scores[i],
        reverse=True
    )[:top_k]

    return [(corpus[i], float(scores[i])) for i in top_indices]

# Run all 8 queries
for q in queries:
    print(f"\nQuery {q['id']}: {q['text']}")

    results = bm25_search(q["text"])

    for rank, (chunk, score) in enumerate(results, 1):
        first_line = chunk["text"].splitlines()[0]
        print(f"\nRank {rank}: {chunk['id']}")
        print(f"Score: {score:.4f}")
        print(f"First line: {first_line}")

    relevant = set(q["relevant"])
    hit = any(chunk["id"] in relevant for chunk, _ in results)
    print(f"\nHit@3: {int(hit)}")
    print("-" * 50)



Query Q-01: What is form GR-305 used for?

Rank 1: GO-08
Score: 15.8255
First line: Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.

Rank 2: GO-12
Score: 8.1953
First line: A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the evidence that was on file; new income information cannot be added through an appeal and needs a new application.

Rank 3: GO-06
Score: 8.0700
First line: Form GR-104 is the application form for the need-based study grant. It is the only way

## B2 — EmbeddingGemma (`google/embeddinggemma-300m`) with `encode_document` / `encode_query`

In [6]:

from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

print("HF_TOKEN loaded:", bool(hf_token))


HF_TOKEN loaded: True


In [7]:

%pip install -q "sentence-transformers>=5.1" "transformers>=4.56"

import torch
from sentence_transformers import SentenceTransformer

# Select device
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Load EmbeddingGemma
embedding_model = SentenceTransformer(
    "google/embeddinggemma-300m",
    token=hf_token,
    device=device
)

print("Model loaded successfully!")

# Encode all 74 documents
documents = [chunk["text"] for chunk in corpus]

document_embeddings = embedding_model.encode_document(
    documents,
    batch_size=4,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Documents encoded:", len(document_embeddings))
print("Embedding shape:", document_embeddings.shape)


Device: cpu


modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

...

3_Dense/model.safetensors: reconstructing file:   0%|          |  0.00B / 9.44MB            

3_Dense/model.safetensors: downloading bytes:           |  0.00B            

Model loaded successfully!


Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Documents encoded: 74
Embedding shape: (74, 768)


In [8]:

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Encode the 8 queries
query_texts = [q["text"] for q in queries]

query_embeddings = embedding_model.encode_query(
    query_texts,
    batch_size=4,
    convert_to_numpy=True,
    show_progress_bar=True
)

# Calculate cosine similarities
similarities = cosine_similarity(
    query_embeddings, document_embeddings
)

embedding_results = {}

# Retrieve top 3 documents for each query
for q, scores in zip(queries, similarities):
    top_indices = np.argsort(scores)[::-1][:3]

    top3 = [
        (corpus[int(i)]["id"], float(scores[i]))
        for i in top_indices
    ]

    embedding_results[q["id"]] = top3

    print(f"\n{q['id']}: {q['text']}")

    for rank, (doc_id, score) in enumerate(top3, 1):
        print(f"Rank {rank}: {doc_id} | Cosine: {score:.4f}")

    relevant = set(q["relevant"])
    hit = any(doc_id in relevant for doc_id, _ in top3)

    print(f"Hit@3: {int(hit)}")
    print("-" * 45)

print("\nEmbeddingGemma search completed!")


Batches:   0%|          | 0/2 [00:00<?, ?it/s]


Q-01: What is form GR-305 used for?
Rank 1: GO-08 | Cosine: 0.6955
Rank 2: GO-12 | Cosine: 0.4609
Rank 3: GO-23 | Cosine: 0.4505
Hit@3: 1
---------------------------------------------

Q-02: My family is poor. Is there financial help for students like me?
Rank 1: GO-01 | Cosine: 0.4446
Rank 2: GO-10 | Cosine: 0.4117
Rank 3: GO-20 | Cosine: 0.4090
Hit@3: 1
---------------------------------------------

Q-03: Өтінішті қашанға дейін тапсыру керек?
Rank 1: GO-14 | Cosine: 0.3992
Rank 2: GO-09 | Cosine: 0.3502
Rank 3: GO-11 | Cosine: 0.3007
Hit@3: 1
---------------------------------------------

Q-04: What is the minimum GPA for the grant?
Rank 1: GO-02 | Cosine: 0.6652
Rank 2: GO-05 | Cosine: 0.6210
Rank 3: GO-17 | Cosine: 0.6134
Hit@3: 1
---------------------------------------------

Q-05: When was Astana renamed Nur-Sultan, and why?
Rank 1: KZ-05 | Cosine: 0.6514
Rank 2: KZ-06 | Cosine: 0.5074
Rank 3: KZ-03 | Cosine: 0.4186
Hit@3: 1
---------------------------------------------

Q-06: W


## My own queries

| Id | Query | relevant | Expected winner, and why |
|---|---|---|---|
| Q-09 | What must be signed by every adult on form GR-210? | GO-07 | BM25 is expected to perform better because the query contains the exact form code GR-210 and specific words from the document. |
| Q-10 | If I temporarily stop studying and return later, what happens to my grant payments? | GO-18 | EmbeddingGemma is expected to perform better because the query paraphrases academic leave and payment suspension without using the exact terminology. |
| Q-11 | What is ak-serke? | KZ-25 | BM25 is expected to perform better because the query contains the rare, exact term "ak-serke", which appears in the relevant document. |
| Q-12 | What is located at 1,691 m (5,548 ft)? | KZ-35 | BM25 is expected to perform better because the query contains the exact numerical values "1,691" and "5,548", which appear in KZ-35. |


In [9]:

# Two additional queries
own_queries = [
    {
        "id": "Q-09",
        "text": "What must be signed by every adult on form GR-210?",
        "relevant": ["GO-07"]
    },
    {
        "id": "Q-10",
        "text": "If I temporarily stop studying and return later, what happens to my grant payments?",
        "relevant": ["GO-18"]
    }
]

# Combine all 10 queries
all_queries = queries + own_queries

print("Total queries:", len(all_queries))

# Test the two new queries
for q in own_queries:
    print(f"\n{q['id']}: {q['text']}")
    relevant = set(q["relevant"])

    # BM25
    bm_results = bm25_search(q["text"], top_k=3)
    print("\nBM25 Top 3:")

    for rank, (chunk, score) in enumerate(bm_results, 1):
        print(f"{rank}. {chunk['id']} | Score: {score:.4f}")

    bm_hit = any(chunk["id"] in relevant for chunk, _ in bm_results)
    print("BM25 Hit@3:", int(bm_hit))

    # EmbeddingGemma
    q_embedding = embedding_model.encode_query(
        [q["text"]],
        convert_to_numpy=True
    )

    scores = cosine_similarity(q_embedding, document_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:3]

    print("\nEmbeddingGemma Top 3:")

    for rank, i in enumerate(top_indices, 1):
        print(f"{rank}. {corpus[i]['id']} | Cosine: {scores[i]:.4f}")

    emb_hit = any(corpus[i]["id"] in relevant for i in top_indices)
    print("Embedding Hit@3:", int(emb_hit))

    print("-" * 50)


Total queries: 10

Q-09: What must be signed by every adult on form GR-210?

BM25 Top 3:
1. GO-07 | Score: 27.5013
2. GO-13 | Score: 13.2595
3. GO-10 | Score: 10.0527
BM25 Hit@3: 1

EmbeddingGemma Top 3:
1. GO-07 | Cosine: 0.6980
2. GO-10 | Cosine: 0.5033
3. GO-13 | Cosine: 0.4337
Embedding Hit@3: 1
--------------------------------------------------

Q-10: If I temporarily stop studying and return later, what happens to my grant payments?

BM25 Top 3:
1. GO-18 | Score: 18.5812
2. GO-13 | Score: 11.2971
3. GO-17 | Score: 9.6880
BM25 Hit@3: 1

EmbeddingGemma Top 3:
1. GO-18 | Cosine: 0.7378
2. GO-17 | Cosine: 0.6861
3. GO-15 | Cosine: 0.5080
Embedding Hit@3: 1
--------------------------------------------------


In [10]:

# Q-11: Exact term retrieval test
q11 = {
    "id": "Q-11",
    "text": "What is ak-serke?",
    "relevant": ["KZ-25"]
}

# Add Q-11 to the full query set
all_queries = [q for q in all_queries if q["id"] != "Q-11"] + [q11]

print("Total queries:", len(all_queries))

# BM25
bm_results = bm25_search(q11["text"], top_k=3)

print("\nBM25 Top 3:")
for rank, (chunk, score) in enumerate(bm_results, 1):
    print(f"{rank}. {chunk['id']} | Score: {score:.4f}")

# EmbeddingGemma
q_embedding = embedding_model.encode_query(
    [q11["text"]],
    convert_to_numpy=True
)

scores = cosine_similarity(q_embedding, document_embeddings)[0]
top_indices = np.argsort(scores)[::-1][:3]

print("\nEmbeddingGemma Top 3:")
for rank, i in enumerate(top_indices, 1):
    print(f"{rank}. {corpus[i]['id']} | Cosine: {scores[i]:.4f}")

# Compare ranks
bm_ids = [chunk["id"] for chunk, _ in bm_results]
emb_ids = [corpus[i]["id"] for i in top_indices]

print("\nBM25 Hit@3:", int("KZ-25" in bm_ids))
print("Embedding Hit@3:", int("KZ-25" in emb_ids))

print("\nBM25 rank:", bm_ids.index("KZ-25") + 1 if "KZ-25" in bm_ids else "Outside top 3")
print("Embedding rank:", emb_ids.index("KZ-25") + 1 if "KZ-25" in emb_ids else "Outside top 3")


Total queries: 11

BM25 Top 3:
1. KZ-25 | Score: 10.0058
2. GO-13 | Score: 4.4900
3. GO-06 | Score: 1.7837

EmbeddingGemma Top 3:
1. KZ-25 | Cosine: 0.4274
2. KZ-24 | Cosine: 0.2792
3. KZ-39 | Cosine: 0.2706

BM25 Hit@3: 1
Embedding Hit@3: 1

BM25 rank: 1
Embedding rank: 1


In [11]:

# Q-12: Exact numerical matching
q12 = {
    "id": "Q-12",
    "text": "What is located at 1,691 m (5,548 ft)?",
    "relevant": ["KZ-35"]
}

# Add Q-12 without duplicates
all_queries = [q for q in all_queries if q["id"] != "Q-12"] + [q12]

print("Total queries:", len(all_queries))
print("\nQ-12:", q12["text"])
print("Relevant document:", q12["relevant"])

# BM25
bm_results = bm25_search(q12["text"], top_k=3)

print("\nBM25 TOP 3")
for rank, (chunk, score) in enumerate(bm_results, 1):
    print(f"{rank}. {chunk['id']} | Score: {score:.4f}")
    print("First line:", chunk["text"].splitlines()[0])

# EmbeddingGemma
q_vector = embedding_model.encode_query(
    [q12["text"]],
    convert_to_numpy=True
)

scores = cosine_similarity(q_vector, document_embeddings)[0]
top_indices = np.argsort(scores)[::-1][:3]

print("\nEMBEDDINGGEMMA TOP 3")
for rank, i in enumerate(top_indices, 1):
    chunk = corpus[int(i)]
    print(f"{rank}. {chunk['id']} | Cosine: {scores[i]:.4f}")
    print("First line:", chunk["text"].splitlines()[0])

# Evaluation
bm_ids = [chunk["id"] for chunk, score in bm_results]
emb_ids = [corpus[int(i)]["id"] for i in top_indices]

bm_rank = bm_ids.index("KZ-35") + 1 if "KZ-35" in bm_ids else 4
emb_rank = emb_ids.index("KZ-35") + 1 if "KZ-35" in emb_ids else 4

print("\nQ-12 COMPARISON")
print("BM25 rank:", bm_rank)
print("Embedding rank:", emb_rank)
print("BM25 Hit@3:", int(bm_rank <= 3))
print("Embedding Hit@3:", int(emb_rank <= 3))
print("BM25 performed better:", bm_rank < emb_rank)


Total queries: 12

Q-12: What is located at 1,691 m (5,548 ft)?
Relevant document: ['KZ-35']

BM25 TOP 3
1. KZ-35 | Score: 23.0649
First line: The Medeu (Kazakh: Медеу [mʲedʲéw]) is an outdoor speed skating and bandy rink. It is located in Medeu Valley on the southeastern outskirts of Almaty, Kazakhstan. Medeu sits 1,691 m (5,548 ft) above sea level. It consists of 10,500 m2 of ice and utilizes a sophisticated freezing and watering system to ensure the quality of the ice.
2. KZ-46 | Score: 10.6153
First line: The lake stretches 1,200 km (750 mi) from north to south, with an average width of 320 km (200 mi). Its gross coverage is 386,400 km2 (149,200 sq mi) and the surface is about 27 m (89 ft) below sea level. Its main freshwater inflow, Europe's longest river, the Volga, enters at the shallow north end. Two deep basins form its central and southern zones. These lead to horizontal differences in temperature, salinity, and ecology. The seabed in the south reaches 1,023 m (3,356 ft) belo

## Results for all ten or more queries: top 3 from each retriever

In [12]:

import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Encode all 10 queries
all_texts = [q["text"] for q in all_queries]

all_embeddings = embedding_model.encode_query(
    all_texts,
    batch_size=4,
    convert_to_numpy=True
)

all_similarities = cosine_similarity(
    all_embeddings, document_embeddings
)

comparison_rows = []
bm25_top3 = {}
embedding_top3 = {}

for q, scores in zip(all_queries, all_similarities):
    qid = q["id"]
    relevant = set(q["relevant"])

    # BM25 results
    bm_results = bm25_search(q["text"], top_k=3)
    bm25_top3[qid] = bm_results
    bm_ids = [chunk["id"] for chunk, score in bm_results]
    bm_hit = int(any(doc_id in relevant for doc_id in bm_ids))

    # EmbeddingGemma results
    top_indices = np.argsort(scores)[::-1][:3]
    emb_results = [
        (corpus[int(i)], float(scores[i]))
        for i in top_indices
    ]
    embedding_top3[qid] = emb_results
    emb_ids = [chunk["id"] for chunk, score in emb_results]
    emb_hit = int(any(doc_id in relevant for doc_id in emb_ids))

    comparison_rows.append({
        "Query": qid,
        "Relevant": ", ".join(sorted(relevant)),
        "BM25 Top 3": ", ".join(bm_ids),
        "BM25 Hit@3": bm_hit,
        "Embedding Top 3": ", ".join(emb_ids),
        "Embedding Hit@3": emb_hit
    })

comparison_df = pd.DataFrame(comparison_rows)

display(comparison_df)

print("\nOVERALL RESULTS")
print("BM25 Hit@3:", comparison_df["BM25 Hit@3"].mean())
print("Embedding Hit@3:", comparison_df["Embedding Hit@3"].mean())


,Query,Relevant,BM25 Top 3,BM25 Hit@3,Embedding Top 3,Embedding Hit@3
0,Q-01,"GO-08, GO-12, GO-23","GO-08, GO-12, GO-06",1,"GO-08, GO-12, GO-23",1
1,Q-02,"GO-01, GO-02, GO-03","KZ-17, GO-20, KZ-25",0,"GO-01, GO-10, GO-20",1
2,Q-03,"GO-09, GO-14","GO-14, KZ-01, KZ-02",1,"GO-14, GO-09, GO-11",1
3,Q-04,GO-02,"GO-05, GO-17, GO-19",0,"GO-02, GO-05, GO-17",1
4,Q-05,KZ-05,"KZ-05, KZ-06, KZ-11",1,"KZ-05, KZ-06, KZ-03",1
5,Q-06,"KZ-37, KZ-38","KZ-37, KZ-38, KZ-03",1,"KZ-37, KZ-38, KZ-10",1
6,Q-07,GO-15,"GO-21, GO-16, GO-01",0,"GO-15, GO-18, GO-01",1
7,Q-08,GO-17,"GO-17, GO-05, GO-04",1,"GO-17, GO-18, GO-15",1
8,Q-09,GO-07,"GO-07, GO-13, GO-10",1,"GO-07, GO-10, GO-13",1
9,Q-10,GO-18,"GO-18, GO-13, GO-17",1,"GO-18, GO-17, GO-15",1



OVERALL RESULTS
BM25 Hit@3: 0.75
Embedding Hit@3: 1.0


In [13]:

# Print required details for all 10 queries

for q in all_queries:
    qid = q["id"]
    print(f"\nQuery {qid}: {q['text']}")

    print("\nEmbeddingGemma Top 3:")
    for rank, (chunk, score) in enumerate(embedding_top3[qid], 1):
        print(f"Rank {rank}: {chunk['id']}")
        print(f"Score: {score:.4f}")
        print(f"First line: {chunk['text'].splitlines()[0]}")

    # BM25 details for the two additional queries
    if qid in ["Q-09", "Q-10"]:
        print("\nBM25 Top 3:")
        for rank, (chunk, score) in enumerate(bm25_top3[qid], 1):
            print(f"Rank {rank}: {chunk['id']}")
            print(f"Score: {score:.4f}")
            print(f"First line: {chunk['text'].splitlines()[0]}")

    print("-" * 50)

print("\nDetailed retrieval results completed!")



Query Q-01: What is form GR-305 used for?

EmbeddingGemma Top 3:
Rank 1: GO-08
Score: 0.6955
First line: Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or to report a change in circumstances. On GR-305 the student states which part of the decision they believe is wrong and which rule in this handbook supports their case.
Rank 2: GO-12
Score: 0.4609
First line: A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read by a panel of three staff members who did not take part in the original decision, and the panel answers within 20 working days. An appeal reviews the decision on the evidence that was on file; new income information cannot be added through an appeal and needs a new application.
Rank 3: GO-23
Score: 0.4505
First line: Если студент не согласен с решением, он может подать апелляцию по ф

In [22]:
print("Q-11 BM25 detailed results")

for rank, (chunk, score) in enumerate(bm25_top3["Q-11"], 1):
    print(f"Rank {rank}: {chunk['id']}")
    print(f"Score: {score:.4f}")
    print("First line:", chunk["text"].splitlines()[0])

Q-11 BM25 detailed results
Rank 1: KZ-25
Score: 10.0058
First line: Beshbarmak is usually made from finely chopped boiled meat, mixed with dough (typically egg noodles) and chyk, an onion sauce. It is typically served on large communal platters, shared between several people, after shorpo, a first course of mutton broth served in bowls called kese. It is also followed by a broth called ak-serke (shorpo mixed with kymyz or ayran), which is thought to help with settling the stomach. Festive beshbarmak can be cooked together with qazı and chuchuk.
Rank 2: GO-13
Score: 4.4900
First line: A student whose household income changes during the year must report it within 30 days by submitting a new GR-210 with proof. The Grant Office decides from the record, not from what a student says in an email or at the front desk: a claim that 'my file has been updated' has no effect until the new documents are on file and have been checked.
Rank 3: GO-06
Score: 1.7837
First line: Form GR-104 is the applic


## B3 — comparison

| Query | BM25 top 3 (ids) | BM25 hit@3 | Embedding top 3 (ids) | Embedding hit@3 |
|---|---|---:|---|---:|
| Q-01 | GO-08, GO-12, GO-06 | 1 | GO-08, GO-12, GO-23 | 1 |
| Q-02 | KZ-17, GO-20, KZ-25 | 0 | GO-01, GO-10, GO-20 | 1 |
| Q-03 | GO-14, KZ-01, KZ-02 | 1 | GO-14, GO-09, GO-11 | 1 |
| Q-04 | GO-05, GO-17, GO-19 | 0 | GO-02, GO-05, GO-17 | 1 |
| Q-05 | KZ-05, KZ-06, KZ-11 | 1 | KZ-05, KZ-06, KZ-03 | 1 |
| Q-06 | KZ-37, KZ-38, KZ-03 | 1 | KZ-37, KZ-38, KZ-10 | 1 |
| Q-07 | GO-21, GO-16, GO-01 | 0 | GO-15, GO-18, GO-01 | 1 |
| Q-08 | GO-17, GO-05, GO-04 | 1 | GO-17, GO-18, GO-15 | 1 |
| Q-09 | GO-07, GO-13, GO-10 | 1 | GO-07, GO-10, GO-13 | 1 |
| Q-10 | GO-18, GO-13, GO-17 | 1 | GO-18, GO-17, GO-15 | 1 |
| Q-11 | KZ-25, GO-13, GO-06 | 1 | KZ-25, KZ-24, KZ-39 | 1 |
| Q-12 | KZ-35, KZ-46, KZ-36 | 1 | KZ-36, KZ-51, KZ-35 | 1 |
| **hit@3 total** | | **9/12** | | **12/12** |

**Both retrievers agree: which query, and why was it easy?**

For Q-09, both retrievers ranked GO-07 first. The query contains the exact form code "GR-210", making the relevant document easy to identify. Both methods achieved Hit@3 = 1.

**BM25 did better: which query, and what exact string did it match that the embedding model blurred?**

For Q-12, BM25 ranked the relevant document KZ-35 first, while EmbeddingGemma ranked it third. BM25 benefited from the exact numerical values "1,691" and "5,548", which appear in KZ-35. EmbeddingGemma also retrieved the correct document, but gave it a lower rank. Therefore, BM25 performed better in ranking, although both methods achieved Hit@3 = 1.

**Embeddings did better: which query, and what could BM25 not match?**

For Q-02, EmbeddingGemma ranked GO-01 first, while BM25 failed to retrieve any relevant document in its top three. The query uses "My family is poor", while the relevant document describes students from low-income households. EmbeddingGemma recognized this semantic connection.

For Q-07, EmbeddingGemma also retrieved GO-15, a Russian-language document answering an English query about bank payments. BM25 could not match the different languages effectively.

**Conclusion:**

EmbeddingGemma achieved Hit@3 = 100% (12/12), while BM25 achieved 75% (9/12). EmbeddingGemma performed better on paraphrased and multilingual queries. However, BM25 demonstrated an advantage in Q-12 by ranking the document with exact numerical matches higher.


---
# Part C — connect retrieval to the model (1.5 pt)

Model: `gpt-5.6-luna` for every call. Top 3 unless stated.

In [14]:

%pip install -q openai tiktoken

import json
from pathlib import Path
from google.colab import userdata
from openai import OpenAI

# Load OpenAI API key securely
api_key = userdata.get("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

MODEL = "gpt-5.6-luna"

# Load Part C questions
repo = Path("/content/css4007-hw3")

with open(repo / "data/qa.json", encoding="utf-8") as f:
    qa_data = json.load(f)

qa_questions = qa_data["questions"]

print("OpenAI API key loaded:", bool(api_key))
print("Model:", MODEL)
print("Part C questions:", len(qa_questions))

for q in qa_questions:
    print(q["id"], "-", q["question"])

# Check Part B variables
required = [
    "corpus",
    "bm25_search",
    "embedding_model",
    "document_embeddings"
]

missing = [name for name in required if name not in globals()]

if missing:
    print("\nPart B variables need to be restored:", missing)
else:
    print("\nPart B retrieval is ready!")


OpenAI API key loaded: True
Model: gpt-5.6-luna
Part C questions: 6
C-01 - How much does a student in income band 2 receive, and when is it paid?
C-02 - My grant application was refused. How long do I have to appeal, and which form do I use?
C-03 - What is the minimum GPA needed to qualify for the need-based grant?
C-04 - When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?
C-05 - When was Narxoz University founded, and what was it called at the start?
C-06 - I am a master's student. How much need-based grant can I get?

Part B retrieval is ready!


## C1 — the complete prompt, printed before it is sent

In [15]:

import numpy as np
import tiktoken
from sklearn.metrics.pairwise import cosine_similarity

# Select question C-01
q = qa_questions[0]
question = q["question"]

# Retrieve top 3 chunks using EmbeddingGemma
q_vector = embedding_model.encode_query(
    [question], convert_to_numpy=True
)

scores = cosine_similarity(q_vector, document_embeddings)[0]
top_indices = np.argsort(scores)[::-1][:3]

# Build numbered context blocks
blocks = []

for rank, i in enumerate(top_indices, 1):
    chunk = corpus[int(i)]
    blocks.append(
        f"[{rank}] Chunk ID: {chunk['id']}\n"
        f"Text: {chunk['text']}"
    )

context_text = "\n\n".join(blocks)

# System instructions
system_message = (
    "Answer the question using only the provided context.\n"
    "If the answer cannot be found in the context, "
    "say that you do not know.\n"
    "Cite the context blocks you used, like [1] or [2][3]."
)

# Build the complete user message
context_part = f"Context:\n{context_text}\n\n"
question_part = f"Question: {question}"
user_message = context_part + question_part

messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_message}
]

# Print everything BEFORE sending
print("===== COMPLETE PROMPT =====")
print("\nSYSTEM MESSAGE:\n")
print(system_message)
print("\nUSER MESSAGE:\n")
print(user_message)

# Count tokens with tiktoken
enc = tiktoken.get_encoding("o200k_base")

system_tokens = len(enc.encode(system_message))
question_tokens = len(enc.encode(question_part))
context_tokens = len(enc.encode(context_part))

total_tokens = system_tokens + question_tokens + context_tokens

print("\n===== TOKEN COUNTS =====")
print("System instructions:", system_tokens)
print("Question:", question_tokens)
print("Retrieved context:", context_tokens)
print("Total input:", total_tokens)

print("\nNumber of retrieved chunks:", len(blocks))

for i, block in enumerate(blocks, 1):
    print(f"Chunk {i} tokens:", len(enc.encode(block)))

print("Context characters:", len(context_text))
print("Context tokens:", len(enc.encode(context_text)))

# Send the prompt to GPT
response = client.chat.completions.create(
    model=MODEL,
    messages=messages
)

print("\n===== MODEL RESPONSE =====")
print(response.choices[0].message.content)

print("\n===== API TOKEN USAGE =====")
print("usage.prompt_tokens:", response.usage.prompt_tokens)


===== COMPLETE PROMPT =====

SYSTEM MESSAGE:

Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

USER MESSAGE:

Context:
[1] Chunk ID: GO-04
Text: The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] Chunk ID: GO-03
Text: The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are not eligible for the need-bas


| Part of the input | Tokens (tiktoken `o200k_base`) |
|---|---:|
| System instructions | 44 |
| Question | 20 |
| Retrieved context | 317 |
| **Total input** | **381** |
| **`usage.prompt_tokens` from the response** | **391** |

- **Number of retrieved chunks:** 3 (GO-04, GO-03, GO-15).
- **Tokens per chunk:** Chunk 1 = 102, Chunk 2 = 116, Chunk 3 = 97.
- **Total context size:** 315 tokens / 1211 characters (excluding the "Context:" label).

**Why do the tiktoken total and `usage.prompt_tokens` differ?**

The tiktoken count is 381, while the API reports 391 prompt tokens. The difference of 10 tokens is due to additional message formatting and role-related overhead included in the API token count.

**What exactly does retrieval add to the model's input?**

Retrieval adds the actual text of the three selected document chunks, together with their chunk IDs, as numbered context blocks. The model receives this context alongside the system instructions and the question. It does not search the original documents itself. The retrieved context provides specific information that the model can use to answer the question accurately and cite its sources.


## C2 — three systems, six questions

In [16]:

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Keep completed answers if the cell is rerun
if "c2_results" not in globals():
    c2_results = {}

c2_retrievals = {}
c2_contexts = {}

rag_system = (
    "Answer the question using only the provided context.\n"
    "If the answer cannot be found in the context, "
    "say that you do not know.\n"
    "Cite the context blocks you used, like [1] or [2][3]."
)

def get_embedding_top3(question):
    query_vector = embedding_model.encode_query(
        [question], convert_to_numpy=True
    )

    scores = cosine_similarity(
        query_vector, document_embeddings
    )[0]

    indices = np.argsort(scores)[::-1][:3]

    return [
        (corpus[int(i)], float(scores[i]))
        for i in indices
    ]

def build_rag_messages(question, retrieved):
    blocks = []

    for rank, (chunk, score) in enumerate(retrieved, 1):
        blocks.append(
            f"[{rank}] Chunk ID: {chunk['id']}\n"
            f"Text: {chunk['text']}"
        )

    context = "\n\n".join(blocks)

    user_message = (
        f"Context:\n{context}\n\n"
        f"Question: {question}"
    )

    messages = [
        {"role": "system", "content": rag_system},
        {"role": "user", "content": user_message}
    ]

    return messages, context

methods = ["LLM only", "BM25 RAG", "Embedding RAG"]

for q in qa_questions:
    qid = q["id"]
    question = q["question"]

    print("\n" + "=" * 65)
    print(f"{qid}: {question}")
    print("REFERENCE:", q["reference"])
    print("=" * 65)

    for method in methods:
        key = (qid, method)

        if method == "LLM only":
            messages = [
                {"role": "user", "content": question}
            ]
            retrieved = []

        elif method == "BM25 RAG":
            retrieved = bm25_search(question, top_k=3)
            messages, context = build_rag_messages(
                question, retrieved
            )
            c2_contexts[key] = context

        else:
            retrieved = get_embedding_top3(question)
            messages, context = build_rag_messages(
                question, retrieved
            )
            c2_contexts[key] = context

        c2_retrievals[key] = retrieved

        # Call GPT only if this answer is not already saved
        if key not in c2_results:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages
            )

            c2_results[key] = {
                "answer": response.choices[0].message.content,
                "prompt_tokens": response.usage.prompt_tokens,
                "completion_tokens": response.usage.completion_tokens
            }

        print(f"\n--- {method} ---")

        if retrieved:
            print("Retrieved chunks:")
            for rank, (chunk, score) in enumerate(retrieved, 1):
                print(
                    f"  {rank}. {chunk['id']} "
                    f"(score={score:.4f})"
                )

        print("ANSWER:")
        print(c2_results[key]["answer"])

        print(
            "Prompt tokens:",
            c2_results[key]["prompt_tokens"]
        )

print("\nC2 COMPLETED")
print("Total model responses:", len(c2_results))



C-01: How much does a student in income band 2 receive, and when is it paid?
REFERENCE: 150,000 tenge for the academic year, in two equal instalments: October and February.

--- LLM only ---
ANSWER:
Which student funding scheme and country do you mean? “Income band 2” is used by several programs, and the amount and payment schedule vary by scheme and academic year.
Prompt tokens: 24

--- BM25 RAG ---
Retrieved chunks:
  1. GO-04 (score=20.2192)
  2. GO-02 (score=18.7422)
  3. GO-01 (score=17.2430)
ANSWER:
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: **75,000 tenge in October** and **75,000 tenge in February**. [1]
Prompt tokens: 394

--- Embedding RAG ---
Retrieved chunks:
  1. GO-04 (score=0.5845)
  2. GO-03 (score=0.4842)
  3. GO-15 (score=0.4506)
ANSWER:
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: **75,000 tenge in October** and **75,000 tenge in Fe

In [17]:

import pandas as pd
from IPython.display import display

# Human evaluation against the six reference answers
evaluation = {
    "C-01": [0, 1, 1],
    "C-02": [0, 1, 1],
    "C-03": [0, 1, 1],
    "C-04": [1, 1, 1],
    "C-05": [1, 1, 1],
    "C-06": [0, 1, 1]
}

methods = ["LLM only", "BM25 RAG", "Embedding RAG"]

rows = []
for qid, marks in evaluation.items():
    row = {"Question": qid}
    for method, mark in zip(methods, marks):
        assert (qid, method) in c2_results
        row[method] = "✓" if mark else "✗"
    rows.append(row)

df_c2 = pd.DataFrame(rows)
display(df_c2)

print("\nAGREEMENT WITH REFERENCE")
for i, method in enumerate(methods):
    correct = sum(marks[i] for marks in evaluation.values())
    print(f"{method}: {correct}/6 ({correct/6:.1%})")


,Question,LLM only,BM25 RAG,Embedding RAG
0,C-01,✗,✓,✓
1,C-02,✗,✓,✓
2,C-03,✗,✓,✓
3,C-04,✓,✓,✓
4,C-05,✓,✓,✓
5,C-06,✗,✓,✓



AGREEMENT WITH REFERENCE
LLM only: 2/6 (33.3%)
BM25 RAG: 6/6 (100.0%)
Embedding RAG: 6/6 (100.0%)



Mark each answer ✓ (agrees with `reference`) or ✗, and give a few words of the answer.

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ Asked for scheme | ✓ 150,000 tenge; Oct/Feb | ✓ 150,000 tenge; Oct/Feb |
| C-02 | ✗ No exact deadline or form | ✓ 14 days; GR-305 | ✓ 14 days; GR-305 |
| C-03 | ✗ Suggested GPA 2.0–2.5 | ✓ GPA 2.67 | ✓ GPA 2.67 |
| C-04 | ✓ October 2017; 2031 | ✓ October 2017; 2031 | ✓ October 2017; 2031 |
| C-05 | ✓ 1963; Alma-Ata Institute | ✓ 1963; Almaty Institute | ✓ 1963; Almaty Institute |
| C-06 | ✗ Generic US grant advice | ✓ Amount unknown; separate scheme | ✓ Amount unknown; separate scheme |
| **Agrees with reference** | **2/6** | **6/6** | **6/6** |

**Note on C-05:** "Alma-Ata" is the historical city-name variant of "Almaty", so the LLM-only answer is accepted as equivalent to the reference.

**Conclusion:**

The LLM-only system agreed with the reference in 2 out of 6 questions. Both BM25 RAG and Embedding RAG achieved 6 out of 6 correct answers. Retrieval provided specific information from the documents, helping the model answer questions about fictional grant rules. For C-06, both RAG systems correctly acknowledged that the grant amount for master's students was not provided in the corpus.



## C3 — question 1 of 3: C-02

**Question:** My grant application was refused. How long do I have to appeal, and which form do I use?

**1. What BM25 retrieved (top 3, ids and scores):**

| Rank | Chunk ID | BM25 Score |
|---|---|---:|
| 1 | GO-05 | 17.2414 |
| 2 | GO-08 | 14.5287 |
| 3 | GO-12 | 13.5735 |

BM25 ranked GO-05 first, even though it contains archived 2025 grant rules rather than the appeal procedure.

**2. What EmbeddingGemma retrieved (top 3, ids and scores):**

| Rank | Chunk ID | Cosine Similarity |
|---|---|---:|
| 1 | GO-08 | 0.6029 |
| 2 | GO-12 | 0.5591 |
| 3 | GO-11 | 0.5343 |

EmbeddingGemma ranked the appeal-related documents GO-08 and GO-12 first.

**3. What was sent to the model (the context blocks):**

The complete numbered context blocks for both RAG systems are printed in the Python output below.

BM25 received GO-05, GO-08 and GO-12.

Embedding RAG received GO-08, GO-12 and GO-11.

Both systems received the same question and system instructions, but different retrieved contexts.

**4. How retrieval changed the answer:**

Without retrieval, the LLM could not identify the exact deadline or form and asked for more information.

Both RAG systems correctly answered that the student has 14 calendar days from the decision letter and must use form GR-305.

Retrieval made the answers more accurate, specific and grounded in the documents.

**5. What was missed, and was it a retrieval problem or a generation problem? Quote the evidence.**

BM25 retrieved an irrelevant archived document (GO-05) in first place. This was a retrieval-quality problem, but it did not cause an incorrect final answer because GO-12 was also included.

Evidence from GO-12: "A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305."

Both RAG systems used the relevant information correctly. Therefore, no generation failure occurred in this example.

The additional Top-K experiment also showed a retrieval failure for C-02. With K = 1, BM25 returned only GO-05, so the model could not provide the appeal deadline or form. With K = 3, GO-12 was included and the model answered correctly. This shows how missing relevant context can lead to an incomplete answer.

In [18]:

# C3: Print actual retrieval evidence from C2
# No new OpenAI API calls

def print_c3_evidence(qid):
    q = next(q for q in qa_questions if q["id"] == qid)

    print("=" * 65)
    print("QUESTION:", qid)
    print(q["question"])
    print("\nREFERENCE:", q["reference"])

    print("\n===== LLM ONLY =====")
    print(c2_results[(qid, "LLM only")]["answer"])

    for method in ["BM25 RAG", "Embedding RAG"]:
        key = (qid, method)

        print("\n" + "=" * 65)
        print(method)
        print("=" * 65)

        print("\nTOP 3 RETRIEVED CHUNKS:")
        for rank, (chunk, score) in enumerate(
            c2_retrievals[key], 1
        ):
            print(
                f"{rank}. {chunk['id']} | "
                f"Score: {score:.4f}"
            )

        print("\nCOMPLETE CONTEXT SENT TO MODEL:")
        print(c2_contexts[key])

        print("\nMODEL ANSWER:")
        print(c2_results[key]["answer"])

print_c3_evidence("C-02")


QUESTION: C-02
My grant application was refused. How long do I have to appeal, and which form do I use?

REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305.

===== LLM ONLY =====
I’m sorry your application was refused. The appeal deadline and form depend on **which grant and country/agency** issued the decision.

Please check the refusal letter or decision email—it should state:

- the **appeal deadline** (often counted from the date you received the decision);
- whether you must request a **review**, submit an **appeal**, or make a **complaint**; and
- the required form or online portal.

Tell me the **grant name, issuing body, and country**—or paste the relevant wording from the refusal letter (remove personal details)—and I can identify the exact deadline and form. If the deadline is approaching, contact the grant office immediately and ask them to confirm the process in writing.

BM25 RAG

TOP 3 RETRIEVED CHUNKS:
1. GO-05 | Score: 17.2414
2. GO-08 |


## C3 — question 2 of 3: C-03

**Question:** What is the minimum GPA needed to qualify for the need-based grant?

**1. What BM25 retrieved (top 3, ids and scores):**

| Rank | Chunk ID | BM25 Score |
|---|---|---:|
| 1 | GO-16 | 15.7985 |
| 2 | GO-02 | 14.0587 |
| 3 | GO-06 | 13.8516 |

BM25 ranked GO-16 first, although GO-02 contains the relevant GPA requirement for the 2026 grant.

**2. What EmbeddingGemma retrieved (top 3, ids and scores):**

| Rank | Chunk ID | Cosine Similarity |
|---|---|---:|
| 1 | GO-02 | 0.6286 |
| 2 | GO-20 | 0.5691 |
| 3 | GO-05 | 0.5451 |

EmbeddingGemma ranked the correct document GO-02 first. However, it also retrieved GO-05, which contains an outdated GPA requirement from 2025.

**3. What was sent to the model (the context blocks):**

The complete numbered context blocks for both RAG systems are printed in the Python output below.

BM25 received GO-16, GO-02 and GO-06.

Embedding RAG received GO-02, GO-20 and GO-05.

Both systems received the same question and system instructions, but different retrieved contexts.

**4. How retrieval changed the answer:**

Without retrieval, the LLM suggested that many grants require a GPA around 2.0–2.5, which did not answer the question correctly.

Both RAG systems correctly answered that the minimum cumulative GPA is 2.67 on a 4.0 scale.

Retrieval provided the exact requirement and improved the accuracy and specificity of the answer.

**5. What was missed, and was it a retrieval problem or a generation problem? Quote the evidence.**

BM25 did not rank the most relevant document first. EmbeddingGemma retrieved an outdated document, GO-05, in third place.

Evidence from GO-02: "their cumulative GPA is at least 2.67 on the 4.0 scale"

Evidence from GO-05: "Under the 2025 scheme the minimum GPA for the grant was 3.0"

This represents a retrieval-quality risk because the model received both current and outdated information.

However, both RAG systems correctly selected the current requirement of 2.67. Therefore, no generation error occurred in this example.


In [19]:
# C3 Question 2: GPA requirement
# Uses existing C2 results without new API calls

print_c3_evidence("C-03")

QUESTION: C-03
What is the minimum GPA needed to qualify for the need-based grant?

REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)

===== LLM ONLY =====
The minimum GPA depends on the specific need-based grant and institution. Many grants require maintaining **satisfactory academic progress**, often around a **2.0–2.5 GPA**, while some have no GPA requirement to apply. Which grant or school are you asking about?

BM25 RAG

TOP 3 RETRIEVED CHUNKS:
1. GO-16 | Score: 15.7985
2. GO-02 | Score: 14.0587
3. GO-06 | Score: 13.8516

COMPLETE CONTEXT SENT TO MODEL:
[1] Chunk ID: GO-16
Text: The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total of all university support a student receives in one academic year is capped at 400,000 tenge. When the cap would be exceeded, the need-based grant is reduced so that the total equals the cap.

[2] Chun


## C3 — question 3 of 3: C-06

**Question:** I am a master's student. How much need-based grant can I get?

**1. What BM25 retrieved (top 3, ids and scores):**

| Rank | Chunk ID | BM25 Score |
|---|---|---:|
| 1 | GO-16 | 16.9958 |
| 2 | GO-01 | 14.1725 |
| 3 | GO-20 | 13.7202 |

BM25 ranked GO-20 third, although it is the most relevant document for the question about master's students.

**2. What EmbeddingGemma retrieved (top 3, ids and scores):**

| Rank | Chunk ID | Cosine Similarity |
|---|---|---:|
| 1 | GO-20 | 0.6664 |
| 2 | GO-16 | 0.6161 |
| 3 | GO-01 | 0.5539 |

EmbeddingGemma ranked GO-20 first, correctly identifying the document that explains the eligibility rules for master's students.

**3. What was sent to the model (the context blocks):**

The complete numbered context blocks for both RAG systems are printed in the Python output below.

BM25 received GO-16, GO-01 and GO-20.

Embedding RAG received GO-20, GO-16 and GO-01.

Both systems received the same question and system instructions, but the retrieved documents appeared in different orders.

**4. How retrieval changed the answer:**

Without retrieval, the LLM gave general information about US graduate financial aid. It did not answer the question according to the provided grant handbook.

Both RAG systems correctly explained that the need-based grant described in the handbook is available only to undergraduate students. Master's students are covered by a separate scheme, and the amount is not specified.

Retrieval made the answers more grounded and prevented the model from inventing an unsupported grant amount.

**5. What was missed, and was it a retrieval problem or a generation problem? Quote the evidence.**

Neither retriever missed the relevant document GO-20. However, BM25 ranked it third, while EmbeddingGemma ranked it first.

Evidence from GO-20: "Master's and doctoral students are supported by a separate scheme, which this handbook does not describe"

This confirms that the grant amount for master's students is not available in the corpus.

The LLM-only system provided unrelated US financial aid information because it lacked the relevant context.

Both RAG systems correctly acknowledged the missing information instead of generating an unsupported amount.

Therefore, there was no generation failure in the RAG systems. The main difference was retrieval ranking quality, with EmbeddingGemma ranking the most relevant document higher.



In [20]:
# C3 Question 3: Master's grant
# Uses existing C2 results without new API calls

print_c3_evidence("C-06")

QUESTION: C-06
I am a master's student. How much need-based grant can I get?

REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.

===== LLM ONLY =====
How much need-based grant you can receive depends mainly on your **country, school, program, income, assets, enrollment status, and family/dependency status**.

If you mean the **United States**:

- **Federal Pell Grant:** Generally **not available for master’s students**; it is mainly for undergraduate students who have not earned a bachelor’s degree.
- **Federal TEACH Grant:** Up to the annual federal limit may be available if you’re in an eligible teaching program and agree to qualifying service requirements.
- **State and university grants:** These are the most common need-based grants for graduate students and vary widely—often from a few hundred dollars to full or partial tuition.
- **FAFSA:** You should still submit the FAFSA. Graduate students are usually t


## Final questions

**1. What is the main difference between BM25 and embedding retrieval?**

BM25 is a keyword-based retrieval method that ranks documents using word frequency and document statistics. Embedding retrieval represents queries and documents as numerical vectors and compares their semantic similarity. BM25 focuses on word matches, while embeddings can recognize similar meanings expressed with different words.

**2. When can BM25 perform better than embeddings?**

BM25 can perform better when queries contain exact numbers, names, codes or rare terms. In Q-12, BM25 ranked KZ-35 first by matching the exact elevation values "1,691" and "5,548", while EmbeddingGemma ranked it third.

**3. When can embeddings perform better than BM25?**

Embeddings perform better when queries use paraphrases, synonyms or different languages. In Q-02, EmbeddingGemma connected "my family is poor" with "low-income household". In Q-07, it retrieved a relevant Russian-language document for an English question.

**4. Why is cosine similarity used with embeddings?**

Cosine similarity measures how closely two vectors point in the same direction. It compares their orientation rather than their magnitude. A higher cosine similarity generally indicates greater semantic similarity between the query and document embeddings.

**5. What information is actually sent to an LLM in a RAG system?**

The LLM receives system instructions, the user's question, and the text of the retrieved document chunks. In our implementation, the retrieved context included three numbered blocks with chunk IDs and their text. The LLM used these blocks to generate an answer and cite the supporting information.

**6. Does the LLM search the original documents itself?**

No. The retrieval system searches the documents first and selects the most relevant chunks. These chunks are then inserted into the LLM's input. The LLM generates its answer using the provided context rather than independently searching the original corpus.

**7. What happens if retrieval returns an irrelevant chunk?**

An irrelevant chunk can distract the model, introduce misleading information, increase token usage or cause an incorrect answer. In C-02, BM25 ranked the archived document GO-05 first. However, the correct appeal information was also retrieved, so the model still answered correctly. If the relevant information is missing, the model may be unable to answer.

**8. Why can increasing Top-K sometimes improve RAG and sometimes make it worse? Report K = 1, 3 and 8 on one question.**

We tested BM25 RAG using C-02: "My grant application was refused. How long do I have to appeal, and which form do I use?"

| Top-K | Retrieved chunks | Prompt tokens | Result |
|---|---|---:|---|
| 1 | GO-05 | 180 | Could not answer |
| 3 | GO-05, GO-08, GO-12 | 360 | Correct: 14 days, GR-305 |
| 8 | GO-05, GO-08, GO-12, GO-06, GO-11, GO-13, KZ-14, GO-07 | 876 | Correct: 14 days, GR-305 |

With K = 1, the model received only an irrelevant archived document and could not answer.

With K = 3, the correct documents were included, allowing the model to identify the 14-day deadline and form GR-305.

With K = 8, the answer remained correct, but prompt tokens increased from 360 to 876.

Increasing Top-K can improve RAG by including missing relevant information. However, too many chunks can introduce irrelevant context, increase costs and potentially confuse the model. In this experiment, K = 3 provided the correct answer using fewer tokens than K = 8.


In [21]:

# Final Question 8: Top-K experiment
# Compare BM25 RAG with K = 1, 3, 8

question = next(
    q["question"] for q in qa_questions
    if q["id"] == "C-02"
)

if "topk_experiment" not in globals():
    topk_experiment = {}

for k in [1, 3, 8]:
    retrieved = bm25_search(question, top_k=k)
    messages, context = build_rag_messages(question, retrieved)

    if k not in topk_experiment:
        if k == 3:
            # Reuse the answer from C2
            answer = c2_results[("C-02", "BM25 RAG")]["answer"]
            tokens = c2_results[("C-02", "BM25 RAG")]["prompt_tokens"]
        else:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages
            )
            answer = response.choices[0].message.content
            tokens = response.usage.prompt_tokens

        topk_experiment[k] = {
            "ids": [chunk["id"] for chunk, _ in retrieved],
            "answer": answer,
            "prompt_tokens": tokens
        }

    result = topk_experiment[k]

    print(f"\n===== TOP-K = {k} =====")
    print("Retrieved chunks:", result["ids"])
    print("Prompt tokens:", result["prompt_tokens"])
    print("Model answer:", result["answer"])

print("\nTop-K experiment completed!")



===== TOP-K = 1 =====
Retrieved chunks: ['GO-05']
Prompt tokens: 180
Model answer: I do not know. The provided context does not specify the appeal deadline or the required form. [1]

===== TOP-K = 3 =====
Retrieved chunks: ['GO-05', 'GO-08', 'GO-12']
Prompt tokens: 360
Model answer: You have **14 calendar days** from the date on the decision letter to appeal. Use **Form GR-305**. [3]

===== TOP-K = 8 =====
Retrieved chunks: ['GO-05', 'GO-08', 'GO-12', 'GO-06', 'GO-11', 'GO-13', 'KZ-14', 'GO-07']
Prompt tokens: 876
Model answer: You have **14 calendar days from the date on the decision letter** to appeal. Use **Form GR-305**, the Grant Office appeal form. [3]

Top-K experiment completed!



**9. Is a vector database required to build a RAG system?**

No. A vector database is not required for a basic RAG system. In this project, we stored embeddings in memory and used cosine similarity to retrieve documents. This approach worked with our 74 document chunks. Vector databases become more useful for larger collections because they support efficient similarity search, indexing and data management.

**10. Which retrieval approach worked better for this corpus, and what evidence supports that? Give hit@3 for both over your ten queries and the C2 agreement counts for both RAG systems.**

EmbeddingGemma performed better overall in this experiment.

We evaluated 12 queries, exceeding the minimum requirement of 10.

- BM25 Hit@3: 9/12 = 75%
- EmbeddingGemma Hit@3: 12/12 = 100%
- BM25 RAG agreement in C2: 6/6
- Embedding RAG agreement in C2: 6/6

EmbeddingGemma retrieved relevant documents more consistently, especially for paraphrased and multilingual queries. However, BM25 ranked the exact numerical match higher in Q-12.

Both RAG systems answered all six C2 questions correctly because they included sufficient relevant information in their retrieved context.

Therefore, EmbeddingGemma achieved better retrieval performance on this corpus, although both RAG systems achieved the same final answer accuracy in C2.


---
# Conclusion


In this homework, I compared BM25 and EmbeddingGemma retrieval and examined how retrieved context affects LLM answers.

In Part A, I calculated BM25 scores and cosine similarity manually. The results showed how stemming improves keyword matching and how cosine similarity measures relationships between numerical vectors.

In Part B, I evaluated both retrieval methods using 12 queries and 74 document chunks. BM25 achieved Hit@3 of 75%, while EmbeddingGemma achieved 100%. EmbeddingGemma performed better with paraphrased and multilingual queries, while BM25 demonstrated an advantage with exact numerical matches.

In Part C, I connected both retrieval methods to GPT and compared them with the LLM-only approach. The LLM-only system answered 2 out of 6 questions correctly, while both RAG systems answered all 6 correctly. The Top-K experiment also showed that retrieving more documents can improve an answer, but increases token usage and may introduce irrelevant information.

Overall, this homework demonstrated that retrieval quality, context selection and prompt construction are important for reliable RAG systems. Embedding retrieval performed better overall in this corpus, but BM25 remains useful for exact matching. The results also showed that providing relevant context helps language models produce more accurate and grounded answers.

As a next step, I would build a hybrid retrieval system combining BM25 and EmbeddingGemma. I would also add filtering for outdated documents and test different Top-K values. This could improve retrieval accuracy, preserve exact word matching, and reduce irrelevant information in the model's context.
